In [1]:
import numpy as np
import pandas as pd



In [2]:
train_df = pd.read_csv("/kaggle/input/tabular-playground-series-dec-2021/train.csv")
test_df = pd.read_csv("/kaggle/input/tabular-playground-series-dec-2021/test.csv")



In [3]:
X = train_df.drop("Cover_Type", axis=1)
Y = train_df["Cover_Type"]
print("X shape:", X.shape, "Y shape:", Y.shape)



X shape: (3600000, 55) Y shape: (3600000,)


In [4]:
from sklearn.model_selection import train_test_split

# Bugfix: stratified split can fail if any class has <2 members (as seen in the error).
# Keep the same idea (small validation set), but fall back to non-stratified split if needed.
VALIDATION_N = 7000  # absolute count

if VALIDATION_N is not None and 0 < VALIDATION_N < len(X):
    try:
        x_train, x_val, y_train, y_val = train_test_split(
            X, Y, test_size=VALIDATION_N, random_state=42, stratify=Y
        )
        print("Used stratified split.")
    except ValueError as e:
        print(
            "Stratified split failed; falling back to non-stratified split. Error was:",
            repr(e),
        )
        x_train, x_val, y_train, y_val = train_test_split(
            X, Y, test_size=VALIDATION_N, random_state=42, shuffle=True
        )
    print("Train/val shapes:", x_train.shape, x_val.shape, y_train.shape, y_val.shape)
else:
    x_train, y_train = X, Y
    x_val, y_val = None, None
    print("No validation split used. Train shape:", x_train.shape, "y:", y_train.shape)

# Consistent feature matrix: drop Id for modeling (do it once, reuse everywhere)
FEATURE_COLS = [c for c in X.columns if c != "Id"]
X_train_model = x_train[FEATURE_COLS]
X_val_model = x_val[FEATURE_COLS] if x_val is not None else None



Stratified split failed; falling back to non-stratified split. Error was: ValueError('The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.')
Train/val shapes: (3593000, 55) (7000, 55) (3593000,) (7000,)


In [5]:
from xgboost import XGBClassifier

# Encode labels to 0..6 for XGBoost multi:softprob
y_train_enc = y_train.astype(np.int32) - 1
y_val_enc = y_val.astype(np.int32) - 1 if y_val is not None else None

base_params = dict(
    n_estimators=239,
    n_jobs=4,
    learning_rate=0.5,
    objective="multi:softprob",
    num_class=7,
    random_state=42,
    eval_metric="mlogloss",  # score-neutral for accuracy; improves training stability/logging
)

fit_kwargs = {}
if X_val_model is not None:
    fit_kwargs["eval_set"] = [(X_val_model, y_val_enc)]
    fit_kwargs["verbose"] = True

try:
    model_xgbc = XGBClassifier(**base_params, tree_method="gpu_hist", gpu_id=0)
    model_xgbc.fit(X_train_model, y_train_enc, **fit_kwargs)
except Exception as e:
    print(
        "GPU training unavailable or failed; falling back to CPU. Error was:", repr(e)
    )
    model_xgbc = XGBClassifier(**base_params, tree_method="hist")
    model_xgbc.fit(X_train_model, y_train_enc, **fit_kwargs)



/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [11:23:27] WARNING: /workspace/src/common/error_msg.cc:45: `gpu_id` is deprecated since2.0.0, use `device` instead. E.g. device=cpu/cuda/cuda:0
  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [11:23:27] WARNING: /workspace/src/context.cc:44: No visible GPU is found, setting device to CPU.
  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [11:23:27] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  warnings.warn(smsg, UserWarning)


GPU training unavailable or failed; falling back to CPU. Error was: XGBoostError('[11:23:27] /workspace/src/tree/updater_gpu_hist.cu:781: Exception in gpu_hist: [11:23:27] /workspace/src/tree/updater_gpu_hist.cu:787: Check failed: ctx_->gpu_id >= 0 (-1 vs. 0) : Must have at least one device\nStack trace:\n  [bt] (0) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0xb27f2a) [0x7fff79b2af2a]\n  [bt] (1) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0xb3e95a) [0x7fff79b4195a]\n  [bt] (2) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0xb483cd) [0x7fff79b4b3cd]\n  [bt] (3) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x460c79) [0x7fff79463c79]\n  [bt] (4) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x461d09) [0x7fff79464d09]\n  [bt] (5) /usr/local/lib/python3.11/dist-packages/xgboost/lib/libxgboost.so(+0x4c54f7) [0x7fff794c84f7]\n  [bt] (6) /usr/local/lib/python3.11/dist-packages/xgboos

In [6]:
test_X_model = test_df[FEATURE_COLS]
y_predict_xgbc_enc = model_xgbc.predict(test_X_model).astype(np.int32)

# Decode back to competition labels 1..7
y_predict_xgbc = y_predict_xgbc_enc + 1
print(
    "Predictions shape:",
    y_predict_xgbc.shape,
    "min/max:",
    int(y_predict_xgbc.min()),
    int(y_predict_xgbc.max()),
)



Predictions shape: (400000,) min/max: 1 7


In [7]:
result = pd.DataFrame(
    {
        "Id": test_df["Id"].astype(np.int64),
        "Cover_Type": y_predict_xgbc.astype(np.int64),
    }
)
print(result.head())
print("Submission df shape:", result.shape, "columns:", list(result.columns))



        Id  Cover_Type
0   814683           1
1  1357371           2
2  2106112           1
3  3483684           1
4  3960754           7
Submission df shape: (400000, 2) columns: ['Id', 'Cover_Type']


In [8]:
out_path = "/kaggle/working/submission.csv"
result.to_csv(out_path, index=False)
print("Done. Wrote", out_path, "with columns:", list(result.columns))

Done. Wrote /kaggle/working/submission.csv with columns: ['Id', 'Cover_Type']
